In [ ]:
import os
import json
import numpy as np
import xarray as xr
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================
test_idx = 259  # Sample index to visualize
lat_dim = 128
num_bins = 64

# Model Config (Must match the training script provided)
architecture = "unet6SR_1x"
base_channels = 128
gn_groups = 8
kernel_size = 3

# Paths
base_dir = Path("/Users/ewellmeyer/Documents/research")
data_dir = base_dir / "HadGEM"
input_file = data_dir / f"GA789_PR_his_rg{lat_dim}.nc"
truth_file = data_dir / f"GA789_dPdK_rg{lat_dim}.nc"

# Updated Ensemble Directory for the "PR -> dP" model
ens_dir = Path(
    f"/Users/ewellmeyer/Documents/research/weights/"
    f"ens_HG789_PR_dPdK_Softmax_{architecture}_ch{base_channels}_k{kernel_size}_{lat_dim}x_dPbins{num_bins}_gn{gn_groups}"
)
norm_stats_path = ens_dir / "norm_stats.json"
bin_info_path   = ens_dir / "born_bins.json"

# Load training split indices
splits_path = ens_dir / "data_splits.npz"
if not splits_path.exists():
    raise FileNotFoundError(f"Data splits file not found at {splits_path}")
splits = np.load(splits_path)
train_indices = splits['train'].tolist()
print(f"Loaded training split: {len(train_indices)} samples")

# Path for cached PPE mean (optional - speeds up repeated runs)
ppe_mean_cache = ens_dir / "ppe_train_mean.nc"

# Find members
member_files = sorted(list(ens_dir.glob(f"ens_*_member*.pth")))
print(f"Found {len(member_files)} ensemble members in:\n{ens_dir}")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# ==============================================================================
# 2. MODEL DEFINITION (1-Channel Input)
# ==============================================================================
from models.unet import Unet6R_1x 

class SoftmaxHead(nn.Module):
    def __init__(self, in_channels, num_bins):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, num_bins, kernel_size=1)
    def forward_components(self, feat):
        logits = self.conv(feat)
        probs  = torch.softmax(logits, dim=1) 
        return probs 

class ProbUNet(nn.Module):
    def __init__(self, input_channels, base_channels, kernel_size, p_drop, num_bins, gn_groups):
        super().__init__()
        self.backbone = Unet6R_1x(
            input_channels=input_channels,
            output_channels=base_channels,
            base_channels=base_channels,
            kernel_size=kernel_size,
            p_drop=p_drop,
            gn_groups=gn_groups,    
        )
        self.head = SoftmaxHead(base_channels, num_bins)

    def forward_components(self, x):
        feat = self.backbone(x)
        return self.head.forward_components(feat)

# ==============================================================================
# 3. COMPUTE PPE TRAINING SET MEAN
# ==============================================================================
def get_ppe_train_mean():
    """Compute or load the mean of the PPE training set."""
    
    # Try to load cached version
    if ppe_mean_cache.exists():
        print(f"Loading cached PPE training mean from {ppe_mean_cache}")
        ds = xr.open_dataset(ppe_mean_cache)
        return ds.dPdK_mean.values
    
    # Otherwise compute from scratch
    print(f"Computing PPE training set mean from {len(train_indices)} samples...")
    ds = xr.open_dataset(truth_file)
    
    # Select training samples and compute mean
    train_data = ds.isel(realization=train_indices)
    ppe_mean = train_data.dPdK.mean(dim='realization').values
    
    # Cache for future use
    print(f"Caching PPE mean to {ppe_mean_cache}")
    ds_mean = xr.Dataset(
        {'dPdK_mean': (['latitude', 'longitude'], ppe_mean)},
        coords={'latitude': ds.latitude, 'longitude': ds.longitude}
    )
    ds_mean.to_netcdf(ppe_mean_cache)
    
    return ppe_mean

# ==============================================================================
# 4. ENSEMBLE INFERENCE (With Advanced Uncertainty)
# ==============================================================================
def get_ensemble_metrics(idx):
    print(f"Preparing sample {idx}...")
    
    # A. Load Stats
    with open(norm_stats_path, "r") as f:
        ns = json.load(f)
    x_mean = np.array(ns["x_mean"], dtype=np.float32).reshape(1,1,1,1)
    x_std  = np.array(ns["x_std"],  dtype=np.float32).reshape(1,1,1,1)
    y_mean = float(ns["y_mean"])
    y_std  = float(ns["y_std"])

    with open(bin_info_path, "r") as f:
        bi = json.load(f)
    bin_centers = np.array(bi["bin_centers_norm"], dtype=np.float32)
    bin_centers_t = torch.tensor(bin_centers, device=device).view(1, -1, 1, 1)

    # B. Load Data
    ds_in = xr.open_dataset(input_file).isel(realization=idx)
    ds_tar = xr.open_dataset(truth_file).isel(realization=idx)
    lats = ds_in.latitude.values
    lons = ds_in.longitude.values

    X_pr = ds_in.PR.values[np.newaxis, np.newaxis, ...] 
    y_true = ds_tar.dPdK.values # (H,W)

    # C. Normalize
    X_norm = (X_pr - x_mean) / x_std
    t_in = torch.tensor(X_norm, dtype=torch.float32, device=device)

    # D. Setup Accumulators
    model = ProbUNet(1, base_channels, kernel_size, 0.0, num_bins, gn_groups).to(device)
    
    # We need to track:
    # 1. Sum of probabilities (to get the averaged distribution)
    # 2. Sum of individual means (to calculate spread of means)
    # 3. Sum of squares of individual means (to calculate spread of means)
    
    accum_probs = torch.zeros((1, num_bins, len(lats), len(lons)), device=device)
    accum_mu    = torch.zeros((1, 1, len(lats), len(lons)), device=device)
    accum_mu_sq = torch.zeros((1, 1, len(lats), len(lons)), device=device)

    # E. Run Inference Loop
    print("Running ensemble inference...")
    with torch.no_grad():
        for m_file in tqdm(member_files):
            ckpt = torch.load(m_file, map_location=device)
            model.load_state_dict(ckpt["model"] if "model" in ckpt else ckpt, strict=False)
            model.eval()
            
            # Get Probs
            probs = model.forward_components(t_in)
            accum_probs += probs
            
            # Get Mean for this member
            mu_mem_norm = (probs * bin_centers_t).sum(dim=1, keepdim=True)
            accum_mu    += mu_mem_norm
            accum_mu_sq += mu_mem_norm ** 2
            
    N = len(member_files)
    
    # F. Process Results
    
    # 1. Averaged Distribution
    avg_probs = accum_probs / N
    
    # 2. Final Prediction (Mean of averaged probs)
    mu_total_norm = (avg_probs * bin_centers_t).sum(dim=1)
    
    # 3. Confidence (Max probability in averaged distribution)
    # shape: (1, H, W)
    conf_norm = avg_probs.max(dim=1).values
    
    # 4. Total Uncertainty (Standard Deviation of averaged distribution)
    # "Soft Target Sigma"
    var_total_norm = (avg_probs * (bin_centers_t - mu_total_norm.unsqueeze(1))**2).sum(dim=1)
    sigma_total_norm = torch.sqrt(var_total_norm)

    # 5. Ensemble Spread (Standard Deviation of the MEMBER MEANS)
    # Var(X) = E[X^2] - (E[X])^2
    mean_of_means = accum_mu / N
    mean_of_sqs   = accum_mu_sq / N
    var_spread_norm = mean_of_sqs - (mean_of_means ** 2)
    # Clamp to 0 to avoid numerical noise producing negative variance
    var_spread_norm = torch.clamp(var_spread_norm, min=0.0)
    sigma_spread_norm = torch.sqrt(var_spread_norm).squeeze(1)

    # G. Denormalize to mm/yr
    pred_mm = mu_total_norm.cpu().numpy().squeeze() * y_std + y_mean
    
    # Uncertainties are scaled by y_std only (no mean add)
    sigma_total_mm  = sigma_total_norm.cpu().numpy().squeeze() * y_std
    sigma_spread_mm = sigma_spread_norm.cpu().numpy().squeeze() * y_std
    
    # Confidence is 0-1, no scaling needed
    conf_map = conf_norm.cpu().numpy().squeeze()

    return lats, lons, X_pr.squeeze(), y_true.squeeze(), pred_mm, conf_map, sigma_total_mm, sigma_spread_mm

# ==============================================================================
# 5. PLOTTING
# ==============================================================================

# Get PPE training mean
ppe_train_mean = get_ppe_train_mean()

# Get ensemble predictions
results = get_ensemble_metrics(test_idx)
lats, lons, P_hist, dP_target, dP_pred, conf_map, sigma_total, sigma_spread = results

# Compute errors
nn_error = np.abs(dP_pred - dP_target)
ppe_error = np.abs(ppe_train_mean - dP_target)

# Skill: positive where NN is better (has lower error)
skill_map = nn_error - ppe_error

# Setup Figure - 2x4 Grid
sns.set_context("paper", font_scale=1.2)
fig = plt.figure(figsize=(24, 10))

proj = ccrs.Robinson(central_longitude=0)
trans = ccrs.PlateCarree()
coast_kw = dict(linewidth=0.5, color='black')

def format_ax(ax, title):
    ax.coastlines(**coast_kw)
    ax.set_title(title, fontsize=13, pad=10, fontweight='bold')
    ax.axis('off')

# --- Row 1: The Basics ---

# 1. Input
ax1 = plt.subplot(2, 4, 1, projection=proj)
im1 = ax1.contourf(lons, lats, P_hist, transform=trans, levels=np.linspace(0, 3500, 21), cmap='YlGnBu', extend='max')
format_ax(ax1, "a. Historical Precipitation (Input)")
cb1 = plt.colorbar(im1, ax=ax1, orientation='horizontal', pad=0.05, shrink=0.9)
cb1.set_label(r"mm yr$^{-1}$", fontsize=10)

# 2. Target
ax2 = plt.subplot(2, 4, 2, projection=proj)
im2 = ax2.contourf(lons, lats, dP_target, transform=trans, levels=np.linspace(-500, 500, 25), cmap='BrBG', extend='both')
format_ax(ax2, "b. True Change (Target)")
cb2 = plt.colorbar(im2, ax=ax2, orientation='horizontal', pad=0.05, shrink=0.9)
cb2.set_label(r"$\Delta P$ (mm yr$^{-1}$ K$^{-1}$)", fontsize=10)

# 3. Prediction
ax3 = plt.subplot(2, 4, 3, projection=proj)
im3 = ax3.contourf(lons, lats, dP_pred, transform=trans, levels=np.linspace(-500, 500, 25), cmap='BrBG', extend='both')
format_ax(ax3, "c. NN Ensemble Prediction (Mean)")
cb3 = plt.colorbar(im3, ax=ax3, orientation='horizontal', pad=0.05, shrink=0.9)
cb3.set_label(r"$\Delta P$ (mm yr$^{-1}$ K$^{-1}$)", fontsize=10)

# 4. Error
ax4 = plt.subplot(2, 4, 4, projection=proj)
im4 = ax4.contourf(lons, lats, nn_error, transform=trans, levels=np.linspace(0, 300, 21), cmap='magma_r', extend='max')
format_ax(ax4, "d. NN Absolute Error")
cb4 = plt.colorbar(im4, ax=ax4, orientation='horizontal', pad=0.05, shrink=0.9)
cb4.set_label(r"Error (mm yr$^{-1}$ K$^{-1}$)", fontsize=10)

# --- Row 2: Diagnostics & Uncertainty ---

# 5. Confidence (Max Probability)
ax5 = plt.subplot(2, 4, 5, projection=proj)
im5 = ax5.contourf(lons, lats, conf_map, transform=trans, levels=np.linspace(0, 0.75, 21), cmap='viridis', extend='max')
format_ax(ax5, "e. Confidence (Max Bin Prob)")
cb5 = plt.colorbar(im5, ax=ax5, orientation='horizontal', pad=0.05, shrink=0.9)
cb5.set_label("Probability [0-1]", fontsize=10)

# 6. Total Uncertainty (Sigma of the soft distribution)
ax6 = plt.subplot(2, 4, 6, projection=proj)
im6 = ax6.contourf(lons, lats, sigma_total, transform=trans, levels=np.linspace(0, 300, 21), cmap='magma_r', extend='max')
format_ax(ax6, "f. Distribution Width ($\sigma_{total}$)")
cb6 = plt.colorbar(im6, ax=ax6, orientation='horizontal', pad=0.05, shrink=0.9)
cb6.set_label(r"Std. Dev (mm yr$^{-1}$ K$^{-1}$)", fontsize=10)

# 7. Ensemble Spread (Disagreement between members)
ax7 = plt.subplot(2, 4, 7, projection=proj)
im7 = ax7.contourf(lons, lats, sigma_spread, transform=trans, levels=np.linspace(0, 300, 21), cmap='magma_r', extend='max')
format_ax(ax7, "g. Ensemble Disagreement ($\sigma_{spread}$)")
cb7 = plt.colorbar(im7, ax=ax7, orientation='horizontal', pad=0.05, shrink=0.9)
cb7.set_label(r"Std. Dev of Means (mm yr$^{-1}$ K$^{-1}$)", fontsize=10)

# 8. Skill vs PPE Mean Baseline
ax8 = plt.subplot(2, 4, 8, projection=proj)
# Use diverging colormap: positive (blue/cool) = NN better, negative (red/warm) = PPE better
skill_levels = np.linspace(-150, 150, 25)
im8 = ax8.contourf(lons, lats, skill_map, transform=trans, levels=skill_levels, cmap='RdBu_r', extend='both')
format_ax(ax8, "h. NN Skill vs PPE Mean")
cb8 = plt.colorbar(im8, ax=ax8, orientation='horizontal', pad=0.05, shrink=0.9)
cb8.set_label(r"$\Delta$ Error (mm yr$^{-1}$ K$^{-1}$) [Blue=NN Better]", fontsize=10)

# Add summary statistics as text annotation
skill_mean = np.mean(skill_map)
skill_median = np.median(skill_map)
pct_better = (skill_map > 0).sum() / skill_map.size * 100
# ax8.text(0.02, 0.98, 
#          f'Mean: {skill_mean:.1f}\n'
#          f'Median: {skill_median:.1f}\n'
#          f'{pct_better:.1f}% improved',
#          transform=ax8.transAxes,
#          fontsize=9, 
#          verticalalignment='top',
#          bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle(f'Sample {test_idx} | {len(member_files)} NN Members | Direct (PR → dPdK)', 
             fontsize=16, fontweight='bold', y=0.98)

plt.tight_layout()
plt.savefig(f'ensemble_analysis_sample_{test_idx}.png', dpi=300, bbox_inches='tight')
print(f"\nFigure saved as: ensemble_analysis_sample_{test_idx}.png")
plt.show()

# Print summary statistics
print("\n" + "="*60)
print("SKILL ANALYSIS (NN vs PPE Training Mean)")
print("="*60)
print(f"Mean Skill (Error Reduction):     {skill_mean:.2f} mm/yr/K")
print(f"Median Skill:                     {skill_median:.2f} mm/yr/K")
print(f"NN Better Than PPE:               {pct_better:.1f}% of pixels")
print(f"Mean NN Error:                    {np.mean(nn_error):.2f} mm/yr/K")
print(f"Mean PPE Error:                   {np.mean(ppe_error):.2f} mm/yr/K")
print(f"Relative Error Reduction:         {(1 - np.mean(nn_error)/np.mean(ppe_error))*100:.1f}%")
print("="*60)